In [1]:
import pandas as pd

store_data = pd.read_csv('store_data.csv')

# convert each row to a list of items (ignoring NaN)
transactions = []
for i in range(len(store_data)):
    basket = []
    for j in range(store_data.shape[1]):
        item = store_data.iloc[i, j]
        if pd.notna(item):
            basket.append(str(item))
    transactions.append(basket)

In [2]:
def support(transactions, itemset):
    count = 0
    for t in transactions:
        ok = True
        for item in itemset:
            if item not in t:
                ok = False
                break
        if ok:
            count += 1
    return count / len(transactions)

In [3]:
def get_frequent_single_items(transactions, min_support):
    item_counts = {}

    # count every item
    for t in transactions:
        for item in t:
            item_counts[item] = item_counts.get(item, 0) + 1
    
    # filter by support
    freq = []
    for item, count in item_counts.items():
        if count / len(transactions) >= min_support:
            freq.append([item])   # store as a list for consistency
    
    return freq


In [4]:
def generate_candidates(Lk):
    Ck1 = []
    n = len(Lk)

    for i in range(n):
        for j in range(i+1, n):
            a = Lk[i]
            b = Lk[j]

            # join only if first k-1 items are same
            if a[:-1] == b[:-1]:
                candidate = a + [b[-1]]
                Ck1.append(candidate)

    return Ck1


In [5]:
def filter_candidates(transactions, candidates, min_support):
    L = []
    for c in candidates:
        if support(transactions, c) >= min_support:
            L.append(c)
    return L

In [6]:
def apriori(transactions, min_support):
    L1 = get_frequent_single_items(transactions, min_support)
    L = [L1]   # list of all frequent sets level-wise

    k = 1
    while True:
        Ck1 = generate_candidates(L[k-1])
        if not Ck1:
            break

        Lk1 = filter_candidates(transactions, Ck1, min_support)
        if not Lk1:
            break

        L.append(Lk1)
        k += 1

    return L


In [7]:
frequent_itemsets = apriori(transactions, min_support=0.004)

for level in frequent_itemsets:
    print(level)

[['burgers'], ['meatballs'], ['eggs'], ['chutney'], ['turkey'], ['avocado'], ['mineral water'], ['milk'], ['energy bar'], ['whole wheat rice'], ['green tea'], ['low fat yogurt'], ['whole wheat pasta'], ['french fries'], ['soup'], ['light cream'], ['shallot'], ['frozen vegetables'], ['spaghetti'], ['pet food'], ['cookies'], ['cooking oil'], ['champagne'], ['salmon'], ['shrimp'], ['chocolate'], ['chicken'], ['honey'], ['oil'], ['fresh tuna'], ['tomatoes'], ['black tea'], ['extra dark chocolate'], ['protein bar'], ['red wine'], ['pasta'], ['pepper'], ['shampoo'], ['rice'], ['sparkling water'], ['ham'], ['body spray'], ['pancakes'], ['grated cheese'], ['white wine'], ['toothpaste'], ['parmesan cheese'], ['fresh bread'], ['ground beef'], ['frozen smoothie'], ['escalope'], ['yams'], ['herb & pepper'], ['tomato sauce'], ['magazines'], ['strawberries'], ['strong cheese'], ['pickles'], ['cake'], ['cottage cheese'], ['hot dogs'], ['brownies'], ['cereals'], ['clothes accessories'], ['bug spray'],

In [10]:
def all_nonempty_subsets(itemset):
    subsets = []
    n = len(itemset)
    # bitmask from 1 to (2^n - 1)
    for mask in range(1, 1 << n):
        subset = []
        for i in range(n):
            if mask & (1 << i):
                subset.append(itemset[i])
        subsets.append(subset)
    return subsets

In [12]:
def generate_rules(transactions, frequent_itemsets, min_confidence):
    rules = []

    for level in frequent_itemsets[1:]:   # skip 1-itemsets
        for itemset in level:
            subsets = all_nonempty_subsets(itemset)

            for A in subsets:
                if len(A) == len(itemset):
                    continue  # skip the whole set
                
                # B = itemset - A
                B = [x for x in itemset if x not in A]

                supp_itemset = support(transactions, itemset)
                supp_A = support(transactions, A)

                conf = supp_itemset / supp_A

                if conf >= min_confidence:
                    rules.append((A, B, conf))

    return rules


In [13]:
rules = generate_rules(transactions, frequent_itemsets, min_confidence=0.2)

for A, B, conf in rules:
    print(f"{A} -> {B}  (confidence={conf})")

['burgers'] -> ['eggs']  (confidence=0.3302752293577982)
['burgers'] -> ['mineral water']  (confidence=0.27981651376146793)
['burgers'] -> ['milk']  (confidence=0.20489296636085627)
['burgers'] -> ['green tea']  (confidence=0.2003058103975535)
['burgers'] -> ['french fries']  (confidence=0.25229357798165136)
['burgers'] -> ['spaghetti']  (confidence=0.24617737003058102)
['ham'] -> ['burgers']  (confidence=0.21105527638190955)
['almonds'] -> ['burgers']  (confidence=0.2565789473684211)
['meatballs'] -> ['eggs']  (confidence=0.21019108280254778)
['meatballs'] -> ['mineral water']  (confidence=0.36942675159235666)
['meatballs'] -> ['milk']  (confidence=0.2611464968152866)
['meatballs'] -> ['spaghetti']  (confidence=0.3057324840764331)
['turkey'] -> ['eggs']  (confidence=0.31130063965884863)
['eggs'] -> ['mineral water']  (confidence=0.28338278931750743)
['mineral water'] -> ['eggs']  (confidence=0.21376608841634023)
['milk'] -> ['eggs']  (confidence=0.23765432098765435)
['low fat yogurt']